# FastTreeSHAP v1/v2 vs. shapiq TreeSHAP-IQ — Census Income Benchmark

This notebook benchmarks **FastTreeSHAP** (algorithms `v1` and `v2`) against
**shapiq**'s `TreeExplainer` (which implements the TreeSHAP-IQ algorithm,
Muschalik et al., AAAI 2024) on the UCI Adult / Census Income dataset.


## What we measure

| Quantity | FastTreeSHAP | shapiq |
|---|---|---|
| Shapley values | `algorithm="v1"`, `"v2"` | `max_order=1, index="SV"` |
| Pairwise interactions | `algorithm="v1"`, `interactions=True` | `max_order=2, index="k-SII"` |

A few notes on the comparison:

* **For order-1 Shapley values, all indices in shapiq reduce to the standard
  Shapley value (SV)**, so the *values* produced by fasttreeshap v1 / v2 and
  shapiq should agree up to floating-point noise. We sanity-check this.
* **For interactions, the layouts differ.** TreeSHAP's interaction values
  (Lundberg et al.) put main effects on the diagonal of an
  $n \times n$ matrix and split each pairwise interaction symmetrically
  between $(i, j)$ and $(j, i)$. shapiq returns an `InteractionValues` object
  keyed by feature tuples. The closest *numerical* analog of TreeSHAP's
  interaction matrix is `index="SII"`; we use `index="k-SII"` (shapiq's
  recommended default) for the timing comparison, since k-SII and SII have
  essentially identical compute cost at order 2.
* **fasttreeshap v2 does not support interactions** (the FastTreeSHAP paper
  only describes v2 for plain SHAP values), so the interaction benchmark only
  compares v1 vs shapiq.

## Parallelism

Both libraries support multi-threaded computation:

* `fasttreeshap.TreeExplainer(..., n_jobs=-1)` — set at construction time.
* `shapiq.TreeExplainer(...).explain_X(X, n_jobs=-1)` — set at explain time.

Both use `joblib` under the hood, so this is a fair fight.

## 1. Setup

In [ ]:
# Install (uncomment if needed)
# !pip install 'numpy<2' fasttreeshap shapiq xgboost lightgbm scikit-learn pandas 

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb

import fasttreeshap
import shapiq

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

print("fasttreeshap:", fasttreeshap.__version__ if hasattr(fasttreeshap, "__version__") else "?")
print("shapiq:      ", shapiq.__version__)

## 2. Load and pre-process Census Income data

We fetch the Adult dataset from OpenML (id 1590, the standard "adult" task)
instead of relying on local files. The pre-processing mirrors the original
notebook: one-hot encode the categorical columns, drop missing-value markers,
and keep the 13 informative columns.

In [ ]:
# Fetch the Adult / Census Income dataset
adult = fetch_openml("adult", version=2, as_frame=True)
X_raw = adult.data
y_raw = adult.target

# Encode label: ">50K" -> 1, "<=50K" -> 0
y = (y_raw == ">50K").astype(int).values

# Drop columns the original benchmark dropped (fnlwgt is kept; capital-gain/loss kept;
# the original used 13 features after dropping the 2 trailing columns from the raw
# UCI text file - OpenML's version already has 14 columns, so we just one-hot encode)
categorical_cols = X_raw.select_dtypes(include=["category", "object"]).columns.tolist()

# Replace "?" markers (encoded as a category) with NaN, then drop those rows for simplicity
X_clean = X_raw.copy()
for col in categorical_cols:
    X_clean[col] = X_clean[col].astype(str).replace("?", np.nan)
mask = X_clean.notna().all(axis=1)
X_clean = X_clean[mask].reset_index(drop=True)
y = y[mask.values]

# One-hot encode categoricals
X = pd.get_dummies(X_clean, columns=categorical_cols, drop_first=False)
# Cast bool dummies to int so xgboost / lightgbm are happy
X = X.astype({c: np.int8 for c in X.columns if X[c].dtype == bool})

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print(f"Training data: {X_train.shape[0]} rows × {X_train.shape[1]} columns")
print(f"Testing  data: {X_test.shape[0]} rows × {X_test.shape[1]} columns")
print(f"Positive class share (train): {y_train.mean():.3f}")

## 3. Helper functions

* `memory_estimate_v2` — same formula as the original notebook; FastTreeSHAP v2
  has a stricter memory footprint than v1 because of its pre-computation tables.
* `run_fasttreeshap` — runs fasttreeshap `num_round` times and reports
  mean ± std wall-clock.
* `run_shapiq` — same pattern for shapiq's `TreeExplainer`.

In [ ]:
def memory_estimate_v2(shap_explainer, num_sample, num_feature, n_jobs):
    """Estimate peak memory of FastTreeSHAP v2 (verbatim from the original benchmark)."""
    max_node = max(shap_explainer.model.num_nodes)
    max_leaves = (max_node + 1) // 2
    max_combinations = 2 ** int(shap_explainer.model.max_depth)
    phi_dim = num_sample * (num_feature + 1) * shap_explainer.model.num_outputs
    n_jobs_eff = os.cpu_count() if n_jobs == -1 else n_jobs
    memory_1 = (max_leaves * max_combinations + phi_dim) * 8 * n_jobs_eff
    memory_2 = max_leaves * max_combinations * shap_explainer.model.values.shape[0] * 8
    memory = min(memory_1, memory_2)
    for unit, divisor in [("B", 1), ("KB", 1024), ("MB", 1024 ** 2), ("GB", 1024 ** 3)]:
        if memory / divisor < 1024 or unit == "GB":
            print(f"Estimated FastTreeSHAP v2 memory: {memory / divisor:.2f} {unit}")
            return

In [ ]:
def run_fasttreeshap(model, sample, *, interactions, algorithm, n_jobs,
                      num_round, num_sample, shortcut=False):
    """Time fasttreeshap `num_round` times. Returns the last result."""
    explainer = fasttreeshap.TreeExplainer(
        model, algorithm=algorithm, n_jobs=n_jobs, shortcut=shortcut
    )
    run_time = np.zeros(num_round)
    out = None
    for i in range(num_round):
        start = time.time()
        out = explainer(sample.iloc[:num_sample], interactions=interactions).values
        run_time[i] = time.time() - start
        print(f"  Round {i + 1}: {run_time[i]:.3f} s")
    label = f"fasttreeshap {algorithm}" + (" (interactions)" if interactions else "")
    print(f"=> {label}: {run_time.mean():.3f} ± {run_time.std():.3f} s")
    return out, run_time

In [ ]:
def run_shapiq(model, sample, *, max_order, n_jobs, num_round, num_sample,
                index=None, class_index=None):
    """Time shapiq's TreeExplainer `num_round` times. Returns the last result."""
    if index is None:
        index = "SV" if max_order == 1 else "k-SII"
    explainer = shapiq.TreeExplainer(
        model=model,
        max_order=max_order,
        min_order=1,
        index=index,
        class_index=class_index,
    )
    X_arr = sample.iloc[:num_sample].values
    run_time = np.zeros(num_round)
    out = None
    for i in range(num_round):
        start = time.time()
        out = explainer.explain_X(X_arr, n_jobs=n_jobs)
        run_time[i] = time.time() - start
        print(f"  Round {i + 1}: {run_time[i]:.3f} s")
    label = f"shapiq (max_order={max_order}, index={index})"
    print(f"=> {label}: {run_time.mean():.3f} ± {run_time.std():.3f} s")
    return out, run_time

In [ ]:
def shapiq_sv_to_array(iv_result, n_features):
    """Convert shapiq's per-sample InteractionValues (order 1) to (n_samples, n_features)."""
    if isinstance(iv_result, list):
        rows = []
        for iv in iv_result:
            # get_n_order_values(1) returns the order-1 (Shapley) values as a 1D array
            row = iv.get_n_order_values(1)
            rows.append(np.asarray(row).ravel())
        return np.vstack(rows)
    # Fallback for batched returns - try .values directly
    return np.asarray(iv_result.values)

In [ ]:
# Global benchmark settings
NUM_SAMPLE_SV = 1000       # samples to explain for SV benchmark
NUM_SAMPLE_INT = 100       # samples to explain for interaction benchmark
NUM_ROUND = 3              # repetitions for mean/std
N_JOBS = -1                # all available cores

## 4. Random Forest (scikit-learn)

We start with `RandomForestClassifier(n_estimators=200, max_depth=8)` — same
configuration as the original benchmark.

In [ ]:
n_estimators = 200
max_depth = 8

rf_model = RandomForestClassifier(
    n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=0
)
rf_model.fit(X_train, y_train)

print(f"AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, rf_model.predict(X_test)):.3f}")

# Tree statistics
_ex = fasttreeshap.TreeExplainer(rf_model)
num_leaves = sum(_ex.model.num_nodes) - sum(sum(_ex.model.children_left > 0))
print(f"Total leaves across the forest: {num_leaves}")
memory_estimate_v2(_ex, NUM_SAMPLE_SV, X_test.shape[1], N_JOBS)

### 4a. Shapley values — fasttreeshap v1 vs v2 vs shapiq

In [ ]:
# fasttreeshap v1
sv_v1, _ = run_fasttreeshap(
    rf_model, X_test, interactions=False, algorithm="v1",
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)

# fasttreeshap v2
sv_v2, _ = run_fasttreeshap(
    rf_model, X_test, interactions=False, algorithm="v2",
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)

# shapiq (max_order=1, index="SV", class_index=1 to match the positive-class slice from fasttreeshap)
sv_shapiq, _ = run_shapiq(
    rf_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

In [ ]:
# Correctness sanity check: order-1 values from all three should agree.
# fasttreeshap returns shape (n_samples, n_features, n_classes) for binary RF.
# We compare to the positive-class slice.
sv_v1_pos = sv_v1[..., 1] if sv_v1.ndim == 3 else sv_v1
sv_v2_pos = sv_v2[..., 1] if sv_v2.ndim == 3 else sv_v2
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])

print(f"v1 vs v2:     max |diff| = {np.max(np.abs(sv_v1_pos - sv_v2_pos)):.2e}")
print(f"v1 vs shapiq: max |diff| = {np.max(np.abs(sv_v1_pos - sv_shapiq_arr)):.2e}")
print(f"v2 vs shapiq: max |diff| = {np.max(np.abs(sv_v2_pos - sv_shapiq_arr)):.2e}")

### 4b. Timing benchmark — Shapley values

Now run each method `NUM_ROUND` times and report mean ± std.

In [ ]:
print("--- fasttreeshap v1 ---")
_, t_rf_v1_sv = run_fasttreeshap(
    rf_model, X_test, interactions=False, algorithm="v1",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- fasttreeshap v2 ---")
_, t_rf_v2_sv = run_fasttreeshap(
    rf_model, X_test, interactions=False, algorithm="v2",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_rf_sq_sv = run_shapiq(
    rf_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

### 4c. Pairwise interactions — fasttreeshap v1 vs shapiq

fasttreeshap v2 does not implement interactions, so it's v1 vs shapiq here.
Sample count is dropped from `NUM_SAMPLE_SV` to `NUM_SAMPLE_INT` because
interaction values cost roughly $O(n_{\text{features}})$ more.

In [ ]:
print("--- fasttreeshap v1 (interactions) ---")
_, t_rf_v1_int = run_fasttreeshap(
    rf_model, X_test, interactions=True, algorithm="v1",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_rf_sq_int = run_shapiq(
    rf_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
    class_index=1,
)

## 5. XGBoost

Same configuration as the original benchmark:
`XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1)`.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1,
    eval_metric="logloss", random_state=0,
)
xgb_model.fit(X_train, y_train)

print(f"AUC: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, xgb_model.predict(X_test)):.3f}")

_ex = fasttreeshap.TreeExplainer(xgb_model)
num_leaves = sum(_ex.model.num_nodes) - sum(sum(_ex.model.children_left > 0))
print(f"Total leaves: {num_leaves}")
memory_estimate_v2(_ex, NUM_SAMPLE_SV, X_test.shape[1], N_JOBS)

### 5a. Shapley values — correctness check

In [ ]:
# XGBoost binary classifier: fasttreeshap returns shape (n_samples, n_features)
# (single output, log-odds). shapiq with class_index=None also gives a single output.
sv_v1, _ = run_fasttreeshap(
    xgb_model, X_test, interactions=False, algorithm="v1",
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_v2, _ = run_fasttreeshap(
    xgb_model, X_test, interactions=False, algorithm="v2",
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_shapiq, _ = run_shapiq(
    xgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)

sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])
print(f"v1 vs v2:     max |diff| = {np.max(np.abs(sv_v1 - sv_v2)):.2e}")
print(f"v1 vs shapiq: max |diff| = {np.max(np.abs(sv_v1 - sv_shapiq_arr)):.2e}")
print(f"v2 vs shapiq: max |diff| = {np.max(np.abs(sv_v2 - sv_shapiq_arr)):.2e}")

### 5b. Timing — Shapley values

In [ ]:
print("--- fasttreeshap v1 ---")
_, t_xgb_v1_sv = run_fasttreeshap(
    xgb_model, X_test, interactions=False, algorithm="v1",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- fasttreeshap v2 ---")
_, t_xgb_v2_sv = run_fasttreeshap(
    xgb_model, X_test, interactions=False, algorithm="v2",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_xgb_sq_sv = run_shapiq(
    xgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)

### 5c. Timing — pairwise interactions

In [ ]:
print("--- fasttreeshap v1 (interactions) ---")
_, t_xgb_v1_int = run_fasttreeshap(
    xgb_model, X_test, interactions=True, algorithm="v1",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_xgb_sq_int = run_shapiq(
    xgb_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)

## 6. LightGBM

Same setup as the original: `LGBMClassifier(n_estimators=200, max_depth=8)`.

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1,
    random_state=0, verbosity=-1,
)
lgb_model.fit(X_train, y_train)

print(f"AUC: {roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, lgb_model.predict(X_test)):.3f}")

_ex = fasttreeshap.TreeExplainer(lgb_model)
num_leaves = sum(_ex.model.num_nodes) - sum(sum(_ex.model.children_left > 0))
print(f"Total leaves: {num_leaves}")
memory_estimate_v2(_ex, NUM_SAMPLE_SV, X_test.shape[1], N_JOBS)

### 6a. Shapley values — correctness check

In [ ]:
sv_v1, _ = run_fasttreeshap(
    lgb_model, X_test, interactions=False, algorithm="v1",
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_v2, _ = run_fasttreeshap(
    lgb_model, X_test, interactions=False, algorithm="v2",
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_shapiq, _ = run_shapiq(
    lgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

# LightGBM binary classifier in fasttreeshap returns shape (n_samples, n_features, 2)
sv_v1_pos = sv_v1[..., 1] if sv_v1.ndim == 3 else sv_v1
sv_v2_pos = sv_v2[..., 1] if sv_v2.ndim == 3 else sv_v2
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])

print(f"v1 vs v2:     max |diff| = {np.max(np.abs(sv_v1_pos - sv_v2_pos)):.2e}")
print(f"v1 vs shapiq: max |diff| = {np.max(np.abs(sv_v1_pos - sv_shapiq_arr)):.2e}")
print(f"v2 vs shapiq: max |diff| = {np.max(np.abs(sv_v2_pos - sv_shapiq_arr)):.2e}")

### 6b. Timing — Shapley values

In [ ]:
print("--- fasttreeshap v1 ---")
_, t_lgb_v1_sv = run_fasttreeshap(
    lgb_model, X_test, interactions=False, algorithm="v1",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- fasttreeshap v2 ---")
_, t_lgb_v2_sv = run_fasttreeshap(
    lgb_model, X_test, interactions=False, algorithm="v2",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_lgb_sq_sv = run_shapiq(
    lgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

### 6c. Timing — pairwise interactions

In [ ]:
print("--- fasttreeshap v1 (interactions) ---")
_, t_lgb_v1_int = run_fasttreeshap(
    lgb_model, X_test, interactions=True, algorithm="v1",
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_lgb_sq_int = run_shapiq(
    lgb_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)

## 7. Summary table

Aggregate everything into one table. `SV` rows are over `NUM_SAMPLE_SV`
samples; `Interaction` rows are over `NUM_SAMPLE_INT` samples.

In [ ]:
def fmt(t):
    return f"{t.mean():.3f} ± {t.std():.3f}"

summary = pd.DataFrame(
    {
        "RF": [
            fmt(t_rf_v1_sv), fmt(t_rf_v2_sv), fmt(t_rf_sq_sv),
            fmt(t_rf_v1_int), "—", fmt(t_rf_sq_int),
        ],
        "XGBoost": [
            fmt(t_xgb_v1_sv), fmt(t_xgb_v2_sv), fmt(t_xgb_sq_sv),
            fmt(t_xgb_v1_int), "—", fmt(t_xgb_sq_int),
        ],
        "LightGBM": [
            fmt(t_lgb_v1_sv), fmt(t_lgb_v2_sv), fmt(t_lgb_sq_sv),
            fmt(t_lgb_v1_int), "—", fmt(t_lgb_sq_int),
        ],
    },
    index=pd.MultiIndex.from_tuples(
        [
            ("SV",          f"fasttreeshap v1 ({NUM_SAMPLE_SV} samples)"),
            ("SV",          f"fasttreeshap v2 ({NUM_SAMPLE_SV} samples)"),
            ("SV",          f"shapiq SV       ({NUM_SAMPLE_SV} samples)"),
            ("Interaction", f"fasttreeshap v1 ({NUM_SAMPLE_INT} samples)"),
            ("Interaction", "fasttreeshap v2 (not supported)"),
            ("Interaction", f"shapiq k-SII   ({NUM_SAMPLE_INT} samples)"),
        ],
        names=["Quantity", "Method"],
    ),
)
summary

## 8. Caveats and things to try

* **Sample sizes**: `NUM_SAMPLE_SV` was reduced from the original notebook's
  `10000` to `1000` because shapiq's TreeSHAP-IQ is typically slower per sample
  than fasttreeshap. Crank it back up if you want longer-running comparisons.
* **shapiq's index choice for interactions**: we used `k-SII`. To benchmark
  the `SII` (closer to TreeSHAP's interaction matrix) or `STII` indices,
  pass `index="SII"` or `index="STII"` to `run_shapiq`. Compute cost at
  `max_order=2` should be essentially the same.
* **Higher-order interactions**: shapiq can compute order-3+ interactions
  (TreeSHAP-IQ's main selling point), which fasttreeshap cannot. Bump
  `max_order=3` in a `run_shapiq` call to see what that costs.
* **Background data**: both `fasttreeshap.TreeExplainer` and
  `shapiq.TreeExplainer` default to the tree-path-dependent perturbation
  (no background dataset). If you want interventional SHAP, pass `data=...`
  to fasttreeshap; shapiq does not currently support interventional perturbation
  in its `TreeExplainer`.
* **`class_index`**: for binary classifiers in scikit-learn and LightGBM,
  fasttreeshap returns both classes (shape `[..., 2]`) and we slice
  `[..., 1]` to match shapiq's positive class. XGBoost returns a single
  log-odds output, so no slicing is needed there.